<a href="https://colab.research.google.com/github/semhfe/4DGaussians-Enhanced/blob/copilot%2Ffix-4dgaussians-enhanced-errors/notebooks/4DGS_Enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 4DGaussians-Enhanced: Google Colab Workflow

Bu notebook, 4D Gaussian Splatting modellerini Google Colab'de eğitmek için eksiksiz bir iş akışı sağlar.

## Özellikler
- ✅ SAM2.1 + YOLO ile otomatik maske oluşturma
- ✅ COLMAP desteği (ham resimlerden kamera poz hesaplama)
- ✅ Maske önizleme ve doğrulama
- ✅ Maske-ağırlıklı loss ile eğitim
- ✅ Eğitim ön ayarları (quick_test, standard, high_quality, fast_motion)
- ✅ Video render ve PLY export

## Donanım Gereksinimleri
- Önerilen: A100 (Colab Pro)
- Minimum: T4 (Ücretsiz) - quick_test preset kullanın

---

## 📦 Cell 1: Kurulum (Installation)

Tüm bağımlılıkları kurar: 4DGaussians, SAM2, COLMAP, C++ submodule'ler

In [3]:
# ============================================================
# CELL 1: INSTALLATION (getcwd HATASI DÜZELTİLDİ)
# ============================================================
import os
import sys
import shutil

PROJECT_DIR = "/content/4DGaussians-Enhanced"

# ==========================================
# 🚨 KRİTİK DÜZELTME: GÜVENLİ BÖLGEYE ÇIK
# ==========================================
# Eğer zaten projenin içindeysek, silmeden önce dışarı çıkmalıyız.
# Yoksa "getcwd: cannot access parent directories" hatası alırız.
os.chdir("/content")
print(f"📍 Güvenli ana dizine geçildi: {os.getcwd()}")

# 1. Temizlik (Temiz bir başlangıç için)
if os.path.exists(PROJECT_DIR):
    print(f"🧹 Eski kurulum temizleniyor: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

# 2. Repo'yu Klonla
print("\n📥 Repo klonlanıyor (Loglar açık)...")
!git clone https://github.com/semhfe/4DGaussians-Enhanced.git

# 3. Proje Klasörüne Gir
os.chdir(PROJECT_DIR)
print(f"📂 Proje dizinine girildi: {os.getcwd()}")

# 4. Doğru Branch'e Geç
print("\n🔀 'copilot/fix-4dgaussians-enhanced-errors' branch'ine geçiliyor...")
!git checkout copilot/fix-4dgaussians-enhanced-errors

# 5. Alt Modülleri İndir (KRİTİK ADIM)
print("\n📦 Alt modüller (Submodules) indiriliyor...")
!git submodule update --init --recursive

# 6. requirements.txt Düzenleme
print("\n🔧 'requirements.txt' düzenleniyor (Torch/MMCV temizliği)...")
!sed -i '/torch/d' requirements.txt
!sed -i '/mmcv/d' requirements.txt
!cat requirements.txt | head -n 5

# 7. Bağımlılıkları Yükleme (LOGLAR AÇIK)
print("\n📦 Python kütüphaneleri kuruluyor (Detaylı çıktı)...")
!pip install matplotlib lpips plyfile pytorch_msssim open3d imageio[ffmpeg] opencv-python
!pip install ultralytics supervision huggingface_hub
# SAM2'yi kaynaktan kur
!pip install "git+https://github.com/facebookresearch/sam2.git"

# 8. Setup Scriptini Çalıştırma
print("\n🔧 Setup scripti çalıştırılıyor (C++ Yamaları, Ninja & COLMAP)...")
if os.path.exists("scripts/colab_setup.py"):
    !python scripts/colab_setup.py
else:
    print("❌ HATA: 'scripts/colab_setup.py' dosyası bulunamadı!")
    print("   Lütfen branch isminin doğru olduğundan emin olun.")
    # Dosya yapısını kontrol et
    if os.path.exists("scripts"):
        print(f"   Mevcut dosyalar: {os.listdir('scripts')}")
    else:
        print("   'scripts' klasörü bile yok! Klonlama hatalı olabilir.")

print("\n" + "="*50)
print("✅ Kurulum tamamlandı! (Lütfen yukarıdaki loglarda 'error' olup olmadığını kontrol edin)")
print("="*50)

📍 Güvenli ana dizine geçildi: /content

📥 Repo klonlanıyor (Loglar açık)...
Cloning into '4DGaussians-Enhanced'...
remote: Enumerating objects: 2702, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 2702 (delta 38), reused 50 (delta 20), pack-reused 2625 (from 1)
Receiving objects: 100% (2702/2702), 66.37 MiB | 16.68 MiB/s, done.
Resolving deltas: 100% (1254/1254), done.
📂 Proje dizinine girildi: /content/4DGaussians-Enhanced

🔀 'copilot/fix-4dgaussians-enhanced-errors' branch'ine geçiliyor...
Branch 'copilot/fix-4dgaussians-enhanced-errors' set up to track remote branch 'copilot/fix-4dgaussians-enhanced-errors' from 'origin'.
Switched to a new branch 'copilot/fix-4dgaussians-enhanced-errors'

📦 Alt modüller (Submodules) indiriliyor...
Submodule 'submodules/depth-diff-gaussian-rasterization' (https://github.com/ingra14m/depth-diff-gaussian-rasterization) registered for path 'submodules/depth-diff-gaussian-rasterization'


In [4]:
# ============================================================
# CELL 1.5: FINAL COMPILATION (C++ Modüllerini Derle)
# ============================================================
import os
import sys

# Proje dizininde olduğumuzdan emin olalım
os.chdir("/content/4DGaussians-Enhanced")

print("🚀 Rasterizer ve Simple-KNN derleniyor (Bu işlem 2-3 dk sürebilir)...")

# 1. Rasterizer Derleme
print("\n📦 Compiling Diff-Gaussian-Rasterization...")
!pip install -e submodules/depth-diff-gaussian-rasterization

# 2. KNN Derleme
print("\n📦 Compiling Simple-KNN...")
!pip install -e submodules/simple-knn

print("\n✅ Derleme tamamlandı! Artık Cell 2'ye geçebilirsiniz.")

🚀 Rasterizer ve Simple-KNN derleniyor (Bu işlem 2-3 dk sürebilir)...

📦 Compiling Diff-Gaussian-Rasterization...
Obtaining file:///content/4DGaussians-Enhanced/submodules/depth-diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  Running setup.py develop for diff_gaussian_rasterization

📦 Compiling Simple-KNN...
Obtaining file:///content/4DGaussians-Enhanced/submodules/simple-knn
  Preparing metadata (setup.py) ... done
  Running setup.py develop for simple_knn

✅ Derleme tamamlandı! Artık Cell 2'ye geçebilirsiniz.


## 📁 Cell 2: Veri Hazırlama (Data Setup)

Google Drive'ı mount eder, veriyi unzip eder ve formatı doğrular.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from google.colab import drive
import os
import zipfile
import shutil

print("="*60)
print("📁 Veri Hazırlama")
print("="*60)

# Step 1: Mount Google Drive
print("\n📂 Google Drive mount ediliyor...")
drive.mount('/content/drive')
print("✅ Drive mount edildi")

# Step 2: Configure paths
# BURADAN DÜZENLEYIN: Veri yollarınızı belirtin
DATA_SOURCE = "/content/drive/MyDrive/4DGS_project/input/added_environment.zip"  # Zip dosyası veya klasör yolu
OUTPUT_BASE = "/content/drive/MyDrive/4DGS_project/output"  # Çıktıların kaydedileceği Drive klasörü

# Local processing paths (faster than Drive)
LOCAL_DATA = "/content/data/my_scene"  # Lokal veri klasörü (işleme için)
LOCAL_OUTPUT = "/content/output"  # Lokal çıktı (eğitim için)

# Step 3: Extract or copy data to local disk
os.makedirs(LOCAL_DATA, exist_ok=True)

if DATA_SOURCE.endswith('.zip'):
    if not os.path.exists(DATA_SOURCE):
        print(f"\n❌ Hata: Zip dosyası bulunamadı: {DATA_SOURCE}")
        print("   Lütfen DATA_SOURCE değişkenini güncelleyin")
    else:
        print(f"\n📦 Zip açılıyor: {DATA_SOURCE}")
        with zipfile.ZipFile(DATA_SOURCE, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_DATA)
        print(f"✅ Zip açıldı: {LOCAL_DATA}")
else:
    if not os.path.exists(DATA_SOURCE):
        print(f"\n❌ Hata: Klasör bulunamadı: {DATA_SOURCE}")
        print("   Lütfen DATA_SOURCE değişkenini güncelleyin")
    else:
        print(f"\n📂 Veri kopyalanıyor: {DATA_SOURCE} -> {LOCAL_DATA}")
        if os.path.exists(LOCAL_DATA):
            shutil.rmtree(LOCAL_DATA)
        shutil.copytree(DATA_SOURCE, LOCAL_DATA)
        print(f"✅ Veri kopyalandı")

# Step 4: Detect data format
print("\n🔍 Veri formatı algılanıyor...")
contents = os.listdir(LOCAL_DATA)
print(f"   İçerik: {contents}")

data_format = None
if 'transforms_train.json' in contents:
    data_format = 'blender'
    print("✅ Format: Blender/NeRF Synthetic")
elif 'sparse' in contents or 'images' in contents:
    data_format = 'colmap'
    print("✅ Format: COLMAP")
elif any('cam' in item for item in contents):
    data_format = 'multicam'
    print("✅ Format: Multi-camera (cam01, cam02, ...)")
elif len([f for f in contents if f.endswith(('.jpg', '.png'))]) > 0:
    data_format = 'raw_images'
    print("✅ Format: Ham resimler (COLMAP gerekli)")
else:
    print("⚠️  Format belirlenemedi. Klasör yapısını kontrol edin.")

# Step 5: Create output directory
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("\n" + "="*60)
print("✅ Veri hazırlama tamamlandı!")
print("="*60)
print(f"\n📁 Lokal veri: {LOCAL_DATA}")
print(f"📁 Lokal çıktı: {LOCAL_OUTPUT}")
print(f"📁 Drive çıktı: {OUTPUT_BASE}")
print(f"\n📊 Format: {data_format}")

if data_format == 'raw_images':
    print("\n⚠️  Ham resimler tespit edildi!")
    print("   Cell 3'ü çalıştırarak COLMAP ile kamera pozlarını hesaplayın")
else:
    print("\n📝 Sonraki adım: Cell 4'ü çalıştırarak maske oluşturun")

📁 Veri Hazırlama

📂 Google Drive mount ediliyor...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mount edildi

📦 Zip açılıyor: /content/drive/MyDrive/4DGS_project/input/added_environment.zip
✅ Zip açıldı: /content/data/my_scene

🔍 Veri formatı algılanıyor...
   İçerik: ['added_environment', '__MACOSX']
⚠️  Format belirlenemedi. Klasör yapısını kontrol edin.

✅ Veri hazırlama tamamlandı!

📁 Lokal veri: /content/data/my_scene
📁 Lokal çıktı: /content/output
📁 Drive çıktı: /content/drive/MyDrive/4DGS_project/output

📊 Format: None

📝 Sonraki adım: Cell 4'ü çalıştırarak maske oluşturun


In [7]:
import os
import shutil

# Assuming LOCAL_DATA is defined in a previous cell
if 'LOCAL_DATA' in locals() or 'LOCAL_DATA' in globals():
    macosx_path = os.path.join(LOCAL_DATA, '__MACOSX')
    if os.path.exists(macosx_path):
        print(f"🧹 '__MACOSX' klasörü siliniyor: {macosx_path}")
        shutil.rmtree(macosx_path)
        print("✅ '__MACOSX' klasörü silindi")
    else:
        print("ℹ️ '__MACOSX' klasörü bulunamadı, silinecek bir şey yok.")
else:
    print("❌ Hata: 'LOCAL_DATA' değişkeni tanımlı değil. Önce 'Cell 2: Veri Hazırlama' kısmını çalıştırın.")

🧹 '__MACOSX' klasörü siliniyor: /content/data/my_scene/__MACOSX
✅ '__MACOSX' klasörü silindi


## 🎯 Cell 3: COLMAP İşleme (Opsiyonel)

**Sadece ham resimleriniz varsa çalıştırın!**

COLMAP ile kamera pozlarını ve sparse point cloud'u hesaplar.

In [10]:
# ============================================================
# CELL 2.5: COLMAP INSTALLATION (ÖN HAZIRLIK)
# ============================================================
# Bu hücreyi Cell 3'ten ÖNCE çalıştırın.
# Sistemde COLMAP yüklü değilse otomatik olarak kurar.
# ============================================================

import shutil
import os

print("🔍 COLMAP kurulumu kontrol ediliyor...")

# COLMAP komutu sistemde var mı bak
if not shutil.which("colmap"):
    print("📦 COLMAP bulunamadı. Kurulum başlatılıyor (1-2 dakika sürebilir)...")
    try:
        # 1. Paket listesini güncelle (Sessiz mod)
        !apt-get update
        # 2. COLMAP'i kur (Sessiz mod, onay istemeden)
        !apt-get install -y colmap
        print("✅ COLMAP başarıyla kuruldu!")
    except Exception as e:
        print(f"❌ Kurulum sırasında hata oluştu: {e}")
        print("👉 İpucu: '!apt-get install -y colmap' komutunu manuel deneyebilirsiniz.")
else:
    print("✅ COLMAP zaten sistemde yüklü, kuruluma gerek yok.")

# Kurulumu doğrula
print("-" * 30)
print("Sürüm Kontrolü:")
!colmap help | head -n 1

🔍 COLMAP kurulumu kontrol ediliyor...
📦 COLMAP bulunamadı. Kurulum başlatılıyor (1-2 dakika sürebilir)...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illi

In [12]:
import os
from PIL import Image
from tqdm.notebook import tqdm

# Görsellerinin şu an bulunduğu asıl konum (Ekran görüntüne göre)
source_root = "/content/data/my_scene/added_environment"

print(f"📂 İşlem başlıyor: {source_root}")

# Klasörleri gez
count = 0
for root, dirs, files in os.walk(source_root):
    # Sadece içinde görsel olan camXX klasörlerini bul
    images = [f for f in files if f.endswith(".png")]

    if images:
        print(f"   -> {os.path.basename(root)} klasöründe {len(images)} görsel dönüştürülüyor...")

        for file in tqdm(images, leave=False):
            png_path = os.path.join(root, file)
            jpg_path = os.path.join(root, file.replace(".png", ".jpg"))

            # Eğer jpg zaten yoksa dönüştür
            if not os.path.exists(jpg_path):
                try:
                    img = Image.open(png_path)
                    # PNG'de şeffaflık (RGBA) olabilir, JPG için RGB'ye çeviriyoruz
                    rgb_img = img.convert('RGB')
                    rgb_img.save(jpg_path, quality=95)
                    count += 1
                except Exception as e:
                    print(f"Hata: {file} dönüştürülemedi. {e}")

print(f"✅ Toplam {count} görsel başarıyla JPG formatına dönüştürüldü.")
print("ℹ️ Not: Script'i çalıştırırken kaynak klasörünü (source path) şu şekilde güncellemeyi unutma:")
print(f"    {source_root}")

📂 İşlem başlıyor: /content/data/my_scene/added_environment
   -> cam06 klasöründe 66 görsel dönüştürülüyor...


  0%|          | 0/66 [00:00<?, ?it/s]

   -> cam05 klasöründe 66 görsel dönüştürülüyor...


  0%|          | 0/66 [00:00<?, ?it/s]

   -> cam08 klasöründe 66 görsel dönüştürülüyor...


...

   -> cam07 klasöründe 66 görsel dönüştürülüyor...


  0%|          | 0/66 [00:00<?, ?it/s]

   -> cam04 klasöründe 66 görsel dönüştürülüyor...


  0%|          | 0/66 [00:00<?, ?it/s]

✅ Toplam 528 görsel başarıyla JPG formatına dönüştürüldü.
ℹ️ Not: Script'i çalıştırırken kaynak klasörünü (source path) şu şekilde güncellemeyi unutma:
    /content/data/my_scene/added_environment


In [13]:
import os

# Temizlenecek ana klasör
target_folder = "/content/data/my_scene/added_environment"

deleted_count = 0

for root, dirs, files in os.walk(target_folder):
    for file in files:
        if file.endswith(".png"):
            file_path = os.path.join(root, file)
            os.remove(file_path)
            deleted_count += 1

print(f"🧹 Temizlik tamamlandı: {deleted_count} adet PNG dosyası silindi.")

🧹 Temizlik tamamlandı: 528 adet PNG dosyası silindi.


asıl olması gereken hem jpg çeviren hem pngleri silen script

In [ ]:
import os
from PIL import Image
from tqdm.notebook import tqdm

# Görsellerin bulunduğu kök klasör
source_root = "/content/data/my_scene/added_environment"

print(f"🔄 Dönüşüm ve Temizlik Başlıyor: {source_root}")

converted_count = 0

for root, dirs, files in os.walk(source_root):
    # Sadece png dosyalarını listele
    images = [f for f in files if f.endswith(".png")]

    if images:
        print(f"   -> {os.path.basename(root)} içinde işlem yapılıyor...")

        for file in tqdm(images, leave=False):
            png_path = os.path.join(root, file)
            jpg_path = os.path.join(root, file.replace(".png", ".jpg"))

            try:
                # 1. Resmi aç ve dönüştür
                img = Image.open(png_path)
                rgb_img = img.convert('RGB')

                # 2. JPG olarak kaydet
                rgb_img.save(jpg_path, quality=95)

                # 3. Kayıt başarılıysa PNG'yi sil (Geri dönüşü yok!)
                if os.path.exists(jpg_path):
                    os.remove(png_path)
                    converted_count += 1

            except Exception as e:
                print(f"❌ Hata: {file} işlenemedi. {e}")

print(f"✅ İşlem tamamlandı. {converted_count} görsel JPG'e çevrildi ve PNG orijinalleri silindi.")

In [17]:
import os
import sys

# --- AYARLAR ---
# Daha önce oluşturduğumuz birleştirilmiş klasör
images_path = "/content/data/my_scene/combined_first_frames"
# Çıktı klasörü
project_path = "/content/data/my_scene/colmap_output"
database_path = os.path.join(project_path, "colmap/database.db")
sparse_output_path = os.path.join(project_path, "sparse")

print("="*60)
print("🛠️ COLMAP Manuel Pipeline (CPU Modu - Güvenli)")
print("="*60)

# 1. Klasörleri Oluştur
os.makedirs(os.path.dirname(database_path), exist_ok=True)
os.makedirs(sparse_output_path, exist_ok=True)

# Veritabanı varsa sil (Temiz başlangıç için)
if os.path.exists(database_path):
    os.remove(database_path)

# --- ADIM 1: Feature Extractor (KRİTİK DÜZELTME BURADA: use_gpu=0) ---
print("\n1️⃣  Özellik Çıkarılıyor (CPU)...")
!colmap feature_extractor \
    --database_path {database_path} \
    --image_path {images_path} \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model OPENCV \
    --SiftExtraction.use_gpu 0

# --- ADIM 2: Matcher (Exhaustive - Az görsel için en iyisi) ---
print("\n2️⃣  Görseller Eşleştiriliyor...")
!colmap exhaustive_matcher \
    --database_path {database_path} \
    --SiftMatching.use_gpu 0

# --- ADIM 3: Mapper (Sparse Model Oluşturma) ---
print("\n3️⃣  3D Nokta Bulutu Oluşturuluyor (Mapper)...")
!colmap mapper \
    --database_path {database_path} \
    --image_path {images_path} \
    --output_path {sparse_output_path}

# --- ADIM 4: Model Dönüştürücü (BIN -> TXT) ---
# 4DGaussians genelde text formatını okur, garanti olsun diye çeviriyoruz.
print("\n4️⃣  Model Dönüştürülüyor...")
# Mapper bazen "0" isimli bir klasör oluşturur. Onu kontrol edelim.
model_path = os.path.join(sparse_output_path, "0")
if not os.path.exists(model_path):
    # Eğer mapper klasör oluşturmadıysa işlem başarısız olmuş olabilir
    print("⚠️ UYARI: Sparse model oluşturulamadı. Görsellerde yeterli örtüşme olmayabilir.")
else:
    !colmap model_converter \
        --input_path {model_path} \
        --output_path {model_path} \
        --output_type TXT
    print(f"\n✅ İŞLEM BAŞARIYLA TAMAMLANDI!")
    print(f"📂 Çıktılar şurada: {model_path}")
    print("📝 Artık 4DGaussians eğitimine (Cell 4 ve sonrası) geçebilirsin.")

🛠️ COLMAP Manuel Pipeline (CPU Modu - Güvenli)

1️⃣  Özellik Çıkarılıyor (CPU)...

Feature extraction

Processed file [1/8]
  Name:            cam05_frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - OPENCV
  Focal Length:    2304.00px
  Features:        1350
Processed file [2/8]
  Name:            cam01_frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - OPENCV
  Focal Length:    2304.00px
  Features:        1319
Processed file [3/8]
  Name:            cam07_frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - OPENCV
  Focal Length:    2304.00px
  Features:        1430
Processed file [4/8]
  Name:            cam02_frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - OPENCV
  Focal Length:    2304.00px
  Features:        1412
Processed file [5/8]
  Name:            cam03_frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - OPENCV
  Focal Length:    2304.00px
  Features:        1422
Processed fi

# **CELL 3.5 colmap için extraview stratejisi**

In [19]:
# 1. CondaColab'ı kur (Kernel Restart Gerektirir)
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:06
🔁 Restarting kernel...


In [1]:
import os
import sys
import shutil

# ==============================================================================
# 🛠️ KURULUM VE AYARLAR
# ==============================================================================

# 1. CUDA Destekli COLMAP'i Conda üzerinden kuruyoruz
print("⚙️  CUDA Destekli COLMAP Kuruluyor (Bu işlem 1-2 dk sürebilir)...")
!conda install -c conda-forge colmap=3.8 cccl -y
print("✅ Kurulum tamamlandı. GPU Testi yapılıyor...")

# 2. Ortam Değişkenleri (A100 optimizasyonu)
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['CUDA_VISIBLE_DEVICES'] = '0' # İlk GPU'yu zorla

# ==============================================================================
# 🚀 FULL GPU PIPELINE (Sparse + Dense)
# ==============================================================================

# Kaynak (Tüm görseller: imageXX + extraXX)
COLMAP_RAW_INPUT_DIR = "/content/drive/MyDrive/4DGS_project/input/work_allviews/images"

# Geçici Çalışma Alanı
COLMAP_WORK_DIR = "/content/colmap_gpu_workspace"
COLMAP_DB = os.path.join(COLMAP_WORK_DIR, "database.db")
COLMAP_SPARSE = os.path.join(COLMAP_WORK_DIR, "sparse")
COLMAP_DENSE = os.path.join(COLMAP_WORK_DIR, "dense")

print("\n" + "="*60)
print("🚀 COLMAP A100 GPU MODU BAŞLIYOR")
print(f"📂 Kaynak: {COLMAP_RAW_INPUT_DIR}")
print("="*60)

# Temiz başlangıç
if os.path.exists(COLMAP_WORK_DIR):
    shutil.rmtree(COLMAP_WORK_DIR)
os.makedirs(COLMAP_WORK_DIR, exist_ok=True)
os.makedirs(COLMAP_SPARSE, exist_ok=True)
os.makedirs(COLMAP_DENSE, exist_ok=True)

# 1. Feature Extraction (GPU)
print("\n1️⃣  Feature Extraction (GPU)...")
!colmap feature_extractor \
    --database_path {COLMAP_DB} \
    --image_path {COLMAP_RAW_INPUT_DIR} \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model OPENCV \
    --SiftExtraction.use_gpu 1

# 2. Matcher (GPU)
print("\n2️⃣  Matcher (GPU)...")
!colmap exhaustive_matcher \
    --database_path {COLMAP_DB} \
    --SiftMatching.use_gpu 1

# 3. Mapper (Sparse Model)
print("\n3️⃣  Mapper (Sparse)...")
!colmap mapper \
    --database_path {COLMAP_DB} \
    --image_path {COLMAP_RAW_INPUT_DIR} \
    --output_path {COLMAP_SPARSE}

# Model yolunu bul (Mapper bazen 0 klasörü açar)
sparse_model_path = os.path.join(COLMAP_SPARSE, "0")
if not os.path.exists(sparse_model_path):
    sparse_model_path = COLMAP_SPARSE

# 4. Image Undistorter (Dense Hazırlığı)
print("\n4️⃣  Image Undistorter (Dense Hazırlığı)...")
!colmap image_undistorter \
    --image_path {COLMAP_RAW_INPUT_DIR} \
    --input_path {sparse_model_path} \
    --output_path {COLMAP_DENSE} \
    --output_type COLMAP \
    --max_image_size 2000

# 5. Patch Match Stereo (Depth Maps - GPU A100 GÜCÜ BURADA LAZIM)
print("\n5️⃣  Patch Match Stereo (Derinlik Haritaları - GPU)...")
# GPU index 0'ı kullan, geom_consistency kaliteyi artırır
!colmap patch_match_stereo \
    --workspace_path {COLMAP_DENSE} \
    --workspace_format COLMAP \
    --PatchMatchStereo.geom_consistency 1 \
    --PatchMatchStereo.gpu_index 0

# 6. Stereo Fusion (Dense Point Cloud Üretimi)
print("\n6️⃣  Stereo Fusion (Dense Cloud - GPU)...")
!colmap stereo_fusion \
    --workspace_path {COLMAP_DENSE} \
    --workspace_format COLMAP \
    --input_type geometric \
    --output_path {COLMAP_DENSE}/fused.ply

# 7. Model Converter (TXT) - Filtreleme scripti için hazırlık
print("\n7️⃣  Sparse Model TXT'ye çevriliyor (Filtreleme için)...")
!colmap model_converter \
    --input_path {sparse_model_path} \
    --output_path {sparse_model_path} \
    --output_type TXT

print("\n🎉 GPU İŞLEMLERİ TAMAMLANDI!")
print(f"☁️  Dense Point Cloud: {COLMAP_DENSE}/fused.ply")
print(f"📄 Sparse Model: {sparse_model_path}")
print("👉 Şimdi 'Adım 2: Filtreleme' kodunu çalıştırabilirsin.")

Streaming output truncated to the last 5000 lines.
 Sweep 3: 0.5736s
 Sweep 4: 1.0728s
Iteration 1: 3.2847s
 Sweep 1: 0.5604s
 Sweep 2: 1.0530s
 Sweep 3: 0.5694s
 Sweep 4: 1.0578s
Iteration 2: 3.2408s
 Sweep 1: 0.5571s
 Sweep 2: 1.0383s
 Sweep 3: 0.5658s
 Sweep 4: 1.0443s
Iteration 3: 3.2057s
 Sweep 1: 0.5538s
 Sweep 2: 1.0285s
 Sweep 3: 0.5618s
 Sweep 4: 1.0329s
Iteration 4: 3.1772s
 Sweep 1: 0.5495s
 Sweep 2: 1.0177s
 Sweep 3: 0.5584s
 Sweep 4: 1.0218s
Iteration 5: 3.1476s
Total: 16.1811s

Writing photometric output for extra_011.png

Processing view 10 / 40 for extra_012.png

Reading inputs...

PatchMatch::Problem
-------------------
ref_image_idx: 14
src_image_idxs: 23 7 21 22 27 26 25 5 18 15 13 12 16 11 24 17 4 6 19 20

PatchMatchOptions
-----------------
max_image_size: -1
gpu_index: 0
depth_min: 2.6918
depth_max: 11.0219
window_radius: 5
window_step: 1
sigma_spatial: 5
sigma_color: 0.2
num_samples: 15
ncc_sigma: 0.6
min_triangulation_angle: 1
incident_angle_sigma: 0.9
num_itera

In [2]:
import os
import shutil

# ==============================================================================
# 🎯 SONUÇLARI TOPLAMA
# ==============================================================================
# GPU Workspace'den gelenler
INPUT_SPARSE_DIR = "/content/colmap_gpu_workspace/sparse/0"
INPUT_DENSE_PLY = "/content/colmap_gpu_workspace/dense/fused.ply"

# Hedef (4DGaussians Proje Klasörü)
FINAL_PROJECT_ROOT = "/content/data/my_scene/colmap_output"
FINAL_SPARSE_DIR = os.path.join(FINAL_PROJECT_ROOT, "sparse/0")

print(f"🧹 Filtreleme ve Taşıma Başlıyor...")

# Klasörleri hazırla
if os.path.exists(FINAL_SPARSE_DIR):
    shutil.rmtree(FINAL_SPARSE_DIR)
os.makedirs(FINAL_SPARSE_DIR, exist_ok=True)

# 1. Images.txt Filtreleme ("extra" silinir)
src_images = os.path.join(INPUT_SPARSE_DIR, "images.txt")
dst_images = os.path.join(FINAL_SPARSE_DIR, "images.txt")

count_kept = 0
count_deleted = 0

with open(src_images, "r") as f_in, open(dst_images, "w") as f_out:
    lines = f_in.readlines()
    # Header
    f_out.writelines([l for l in lines if l.startswith("#")])

    data_lines = [l for l in lines if not l.startswith("#")]
    i = 0
    while i < len(data_lines):
        line1 = data_lines[i]
        line2 = data_lines[i+1]
        name = line1.split()[-1]

        if "extra" in name.lower():
            count_deleted += 1
        else:
            f_out.write(line1)
            f_out.write(line2)
            count_kept += 1
        i += 2

print(f"   ✅ Sparse Model Filtrelendi: {count_kept} Ana, {count_deleted} Extra silindi.")

# 2. Diğer Dosyaları Kopyala
shutil.copy(os.path.join(INPUT_SPARSE_DIR, "cameras.txt"), FINAL_SPARSE_DIR)
shutil.copy(os.path.join(INPUT_SPARSE_DIR, "points3D.txt"), FINAL_SPARSE_DIR)

# 3. Dense PLY Kopyala (Burası kritik, artık elimizde gerçek dense var!)
final_ply_path = os.path.join(FINAL_PROJECT_ROOT, "dense_point_cloud.ply")
if os.path.exists(INPUT_DENSE_PLY):
    shutil.copy(INPUT_DENSE_PLY, final_ply_path)
    print(f"   ☁️  Dense Point Cloud (fused.ply) kopyalandı!")
else:
    print("❌ HATA: Dense PLY bulunamadı!")

# 4. BIN Dönüşümü
print("   🔄 .bin formatına dönüştürülüyor...")
!colmap model_converter --input_path {FINAL_SPARSE_DIR} --output_path {FINAL_SPARSE_DIR} --output_type BIN

print("\n🎉 HER ŞEY HAZIR! Training'e geçebilirsin.")

🧹 Filtreleme ve Taşıma Başlıyor...
   ✅ Sparse Model Filtrelendi: 8 Ana, 32 Extra silindi.
   ☁️  Dense Point Cloud (fused.ply) kopyalandı!
   🔄 .bin formatına dönüştürülüyor...

🎉 HER ŞEY HAZIR! Training'e geçebilirsin.


In [3]:
import os
import struct

# ==============================================================================
# 🔍 KONUM AYARLARI
# ==============================================================================
TARGET_ROOT = "/content/data/my_scene/colmap_output"
SPARSE_DIR = os.path.join(TARGET_ROOT, "sparse/0")
DENSE_PLY = os.path.join(TARGET_ROOT, "dense_point_cloud.ply")

print(f"🕵️‍♂️ DOĞRULAMA BAŞLATILIYOR: {TARGET_ROOT}")
print("="*60)

# ------------------------------------------------------------------------------
# 1. IMAGES.TXT İÇERİK KONTROLÜ (Gözle Görünür Kanıt)
# ------------------------------------------------------------------------------
images_txt = os.path.join(SPARSE_DIR, "images.txt")

if not os.path.exists(images_txt):
    print("❌ KRİTİK HATA: images.txt bulunamadı! İşlem başarısız olmuş.")
else:
    print(f"📖 {images_txt} okunuyor...\n")

    with open(images_txt, "r") as f:
        lines = f.readlines()

    # Yorum satırlarını geç
    data_lines = [l for l in lines if not l.startswith("#")]

    # İstatistikler
    # COLMAP txt formatında her resim 2 satırdır (Parametreler + Noktalar)
    total_entries = len(data_lines) // 2

    print(f"   📊 Toplam Kamera Sayısı: {total_entries}")

    # İsim Kontrolü
    clean_count = 0
    dirty_count = 0
    sample_names = []

    i = 0
    while i < len(data_lines):
        line = data_lines[i]
        parts = line.strip().split()
        # Image ID, Q, T, Camera ID, NAME
        # Name en son parçadır
        img_name = parts[-1]

        if "extra" in img_name.lower():
            dirty_count += 1
            print(f"   ❌ ALARM: 'extra' dosya bulundu -> {img_name}")
        else:
            clean_count += 1
            if len(sample_names) < 5: # İlk 5 tanesini örnek al
                sample_names.append(img_name)

        i += 2

    print(f"   ✅ Temiz (Ana) Dosya Sayısı: {clean_count}")
    print(f"   🗑️  Tespit Edilen 'Extra': {dirty_count} (0 olmalı)")

    print("\n   🔎 Örnek Dosya İsimleri (İlk 5):")
    for name in sample_names:
        print(f"      -> {name}")

# ------------------------------------------------------------------------------
# 2. BIN DOSYASI KONTROLÜ (Training İçin Şart)
# ------------------------------------------------------------------------------
print("\n" + "-"*40)
expected_bins = ["images.bin", "cameras.bin", "points3D.bin"]
missing_bins = []

for b in expected_bins:
    if not os.path.exists(os.path.join(SPARSE_DIR, b)):
        missing_bins.append(b)

if missing_bins:
    print(f"❌ EKSİK BIN DOSYALARI: {missing_bins}")
    print("   Training başlamaz!")
else:
    print("✅ Tüm .bin dosyaları mevcut (images.bin, cameras.bin, points3D.bin)")
    # Basit bir byte kontrolü yapalım (Boş mu?)
    size_mb = os.path.getsize(os.path.join(SPARSE_DIR, "points3D.bin")) / (1024*1024)
    print(f"   💾 Sparse Model Boyutu (points3D.bin): {size_mb:.2f} MB")

# ------------------------------------------------------------------------------
# 3. DENSE PLY KONTROLÜ
# ------------------------------------------------------------------------------
print("\n" + "-"*40)
if os.path.exists(DENSE_PLY):
    size_mb = os.path.getsize(DENSE_PLY) / (1024*1024)
    print(f"✅ Dense Point Cloud MEVCUT: {DENSE_PLY}")
    print(f"   ⚖️  Dosya Boyutu: {size_mb:.2f} MB")

    if size_mb < 1:
        print("   ⚠️ UYARI: Dense cloud çok küçük (<1MB). İçeriği boş olabilir.")
    else:
        print("   👍 Boyut makul görünüyor.")
else:
    print("❌ HATA: dense_point_cloud.ply bulunamadı!")

print("="*60)
if dirty_count == 0 and not missing_bins and os.path.exists(DENSE_PLY):
    print("🟢 ONAYLANDI: Veri seti %100 temiz ve eğitime hazır.")
else:
    print("🔴 BAŞARISIZ: Lütfen hataları kontrol edin.")

🕵️‍♂️ DOĞRULAMA BAŞLATILIYOR: /content/data/my_scene/colmap_output
📖 /content/data/my_scene/colmap_output/sparse/0/images.txt okunuyor...

   📊 Toplam Kamera Sayısı: 8
   ✅ Temiz (Ana) Dosya Sayısı: 8
   🗑️  Tespit Edilen 'Extra': 0 (0 olmalı)

   🔎 Örnek Dosya İsimleri (İlk 5):
      -> image08.png
      -> image07.png
      -> image06.png
      -> image05.png
      -> image04.png

----------------------------------------
✅ Tüm .bin dosyaları mevcut (images.bin, cameras.bin, points3D.bin)
   💾 Sparse Model Boyutu (points3D.bin): 0.30 MB

----------------------------------------
✅ Dense Point Cloud MEVCUT: /content/data/my_scene/colmap_output/dense_point_cloud.ply
   ⚖️  Dosya Boyutu: 8.64 MB
   👍 Boyut makul görünüyor.
🟢 ONAYLANDI: Veri seti %100 temiz ve eğitime hazır.


## 🎭 Cell 4: SAM2 Maske Oluşturma

YOLO + SAM2.1 ile otomatik maske oluşturur.

In [3]:
import os
import sys
import shutil

print("="*60)
print("🧹 TEMİZLİK VE RESMİ KURULUM (Facebook Research)")
print("="*60)

# 1. Eski/Hatalı kurulumları temizle
print("1️⃣  Eski kurulumlar kaldırılıyor...")
!pip uninstall -y sam2
!rm -rf /content/sam2  # Eski repo kalıntılarını sil

# 2. Resmi Repoyu Çek
print("\n2️⃣  Resmi SAM2 Reposu indiriliyor...")
!git clone https://github.com/facebookresearch/sam2.git /content/sam2

# 3. Bağımlılıkları ve Kütüphaneyi Kur (-e flag'i ile)
# -e (editable) modu, config dosyalarının doğru algılanması için kritiktir.
print("\n3️⃣  Kütüphane 'Editable Mode' ile kuruluyor...")
%cd /content/sam2
!pip install -e .
%cd /content

# 4. Checkpoint (Model Ağırlığı) Kontrolü
# Config dosyasına ihtiyacımız yok, çünkü temiz kurulumda kütüphanenin içinde zaten var.
# Sadece model ağırlığını (pt dosyasını) indireceğiz.
print("\n4️⃣  Model dosyaları kontrol ediliyor...")
CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
MODEL_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
MODEL_PATH = os.path.join(CHECKPOINT_DIR, "sam2.1_hiera_large.pt")

if not os.path.exists(MODEL_PATH):
    print("   ⬇️  Model ağırlıkları indiriliyor...")
    !wget -q {MODEL_URL} -O {MODEL_PATH}
else:
    print("   ✅ Model ağırlıkları zaten mevcut.")

print("\n✅ KURULUM TAMAMLANDI.")
print("   Artık 'yama' yapmadan sistemin kendi configlerini kullanabiliriz.")

🧹 TEMİZLİK VE RESMİ KURULUM (Facebook Research)
1️⃣  Eski kurulumlar kaldırılıyor...

2️⃣  Resmi SAM2 Reposu indiriliyor...
Cloning into '/content/sam2'...
remote: Enumerating objects: 1070, done.
remote: Total 1070 (delta 0), reused 0 (delta 0), pack-reused 1070 (from 1)
Receiving objects: 100% (1070/1070), 128.11 MiB | 14.79 MiB/s, done.
Resolving deltas: 100% (381/381), done.

3️⃣  Kütüphane 'Editable Mode' ile kuruluyor...
/content/sam2
Obtaining file:///content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
  Created wheel for SAM-2: filename=sam_2-1.0-0.editable-cp311-cp311-linux_x86_64.whl size=13851 sha256=eab84e5f67b0912b82543036179f02d33322c1f1ab6c86a2a6ebc0c2a9e2f092
  Stored in directory: /tmp/pip-ephem-wheel-cache-b8_8a90w/wheels/76/dc/37/006d341f6080de50

/content

4️⃣  Model dosyaları kontrol ediliyor...
   ✅ Model ağırlıkları zaten mevcut.

✅ KURULUM TAMAMLANDI.
   Artık 'yama' yapmadan sistemin kendi configlerini kullanabiliriz.


In [4]:
import sys
import os

# ==========================================================
# 🚑 DÜZELTME: Hafızadan silinen değişkeni tekrar tanımlıyoruz
# Maskelerin görsellerin yanına üretilmesi için doğru yol:
LOCAL_DATA = "/content/data/my_scene/added_environment"
# ==========================================================

sys.path.insert(0, '/content/4DGaussians-Enhanced')

from utils.sam2_utils import generate_masks_for_scene

print("="*60)
print("🎭 SAM2 Maske Oluşturma")
print("="*60)

# SAM2 Konfigürasyonu
DETECTION_PROMPT = "person,human"  # @param {type:"string"}

# Model boyutu - büyük = daha iyi kalite ama yavaş
MODEL_SIZE = "large"  # @param ["tiny", "small", "base", "large"]

# Güven eşiği - yüksek = daha katı tespit
CONFIDENCE_THRESHOLD = 0.5  # @param {type:"slider", min:0.1, max:0.9, step:0.05}

# Her N frame'de bir işle - 1 = tüm frame'ler
EVERY_N_FRAMES = 1  # @param {type:"integer"}

print(f"\n🎯 Parametreler:")
print(f"   Prompt: {DETECTION_PROMPT}")
print(f"   Model: {MODEL_SIZE}")
print(f"   Eşik: {CONFIDENCE_THRESHOLD}")
print(f"   Her N frame: {EVERY_N_FRAMES}")
print(f"   📂 Kaynak: {LOCAL_DATA}")

# Maske oluştur
# Not: Checkpoint yolu condacolab sonrası değişmiş olabilir,
# ama genelde /content/checkpoints altında durur.
stats = generate_masks_for_scene(
    source_path=LOCAL_DATA,
    mask_folder="masks",
    prompt=DETECTION_PROMPT,
    threshold=CONFIDENCE_THRESHOLD,
    every_n=EVERY_N_FRAMES,
    model_size=MODEL_SIZE,
    device="cuda",
    checkpoint_dir="/content/checkpoints"
)

print("\n" + "="*60)
print("✅ Maske oluşturma tamamlandı!")
print("="*60)
print(f"\n📊 İstatistikler: {stats}")
print("\n📝 Sonraki adım: Cell 5 ile maskeleri önizleyin")

🎭 SAM2 Maske Oluşturma

🎯 Parametreler:
   Prompt: person,human
   Model: large
   Eşik: 0.5
   Her N frame: 1
   📂 Kaynak: /content/data/my_scene/added_environment
🎭 Initializing SAM2.1 Mask Generator...
   Prompt: person,human
   Model: large
   Confidence threshold: 0.5
   Processing every 1 frame(s)

📦 Loading SAM2.1 model: large
   Config: /content/checkpoints/sam2.1_hiera_l.yaml
   Checkpoint: /content/checkpoints/sam2.1_hiera_large.pt
❌ Error loading SAM2 model: Cannot find primary config 'content/checkpoints/sam2.1_hiera_l.yaml'. Check that it's in your config search path.

Config search path:
	provider=hydra, path=pkg://hydra.conf
	provider=main, path=pkg://sam2
	provider=schema, path=structured://
📦 Loading YOLO model...
✅ YOLO model loaded successfully

✅ Maske oluşturma tamamlandı!

📊 İstatistikler: {'error': 'Failed to load SAM2 model'}

📝 Sonraki adım: Cell 5 ile maskeleri önizleyin


## 👀 Cell 5: Maske Önizleme

Oluşturulan maskeleri kontrol edin.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import glob
from ipywidgets import interact, IntSlider

print("="*60)
print("👀 Maske Önizleme")
print("="*60)

# Kamera klasörlerini bul
cam_folders = sorted(glob.glob(os.path.join(LOCAL_DATA, "cam*")))
if not cam_folders:
    # Tek klasör yapısı (images/)
    images_folder = os.path.join(LOCAL_DATA, "images")
    if os.path.exists(images_folder):
        cam_folders = [images_folder]

if not cam_folders:
    print("❌ Kamera klasörü bulunamadı")
else:
    def preview_mask(camera_idx=0, frame_idx=0):
        cam_folder = cam_folders[camera_idx]
        cam_name = os.path.basename(cam_folder)

        # Frame dosyalarını bul
        frame_files = sorted(glob.glob(os.path.join(cam_folder, "frame_*.jpg")))
        frame_files.extend(sorted(glob.glob(os.path.join(cam_folder, "frame_*.png"))))
        frame_files.extend(sorted(glob.glob(os.path.join(cam_folder, "*.jpg"))))
        frame_files.extend(sorted(glob.glob(os.path.join(cam_folder, "*.png"))))
        frame_files = sorted(list(set(frame_files)))

        if frame_idx >= len(frame_files):
            print(f"Frame {frame_idx} bulunamadı (toplam {len(frame_files)} frame)")
            return

        frame_path = frame_files[frame_idx]
        frame_name = os.path.basename(frame_path)

        # Maske yolunu bul
        if "frame_" in frame_name:
            mask_name = frame_name.replace("frame_", "mask_").replace(".jpg", ".png")
        else:
            mask_name = f"mask_{frame_name}".replace(".jpg", ".png")

        mask_path = os.path.join(cam_folder, "masks", mask_name)

        # Resmi yükle
        image = np.array(Image.open(frame_path).convert('RGB'))

        # Maskeyi yükle
        if os.path.exists(mask_path):
            mask = np.array(Image.open(mask_path).convert('L'))
            # Overlay oluştur
            overlay = image.copy()
            # Foreground'u yeşile boya
            overlay[:,:,1] = np.where(mask > 128, np.minimum(overlay[:,:,1] + 50, 255), overlay[:,:,1])
        else:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
            overlay = image
            print(f"⚠️  Maske bulunamadı: {mask_path}")

        # Görselleştir
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(image)
        axes[0].set_title(f"{cam_name} - Frame {frame_idx}\nOrijinal")
        axes[0].axis('off')

        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title("Maske\n(Beyaz=Ön plan, Siyah=Arka plan)")
        axes[1].axis('off')

        axes[2].imshow(overlay)
        axes[2].set_title("Overlay\n(Yeşil = Ön plan)")
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

        # İstatistikler
        fg_pixels = np.sum(mask > 128)
        total_pixels = mask.size
        fg_percent = 100 * fg_pixels / total_pixels
        print(f"📊 Ön plan kapsamı: {fg_percent:.1f}%")

    # Frame sayısını bul
    first_cam_frames = glob.glob(os.path.join(cam_folders[0], "*.jpg"))
    first_cam_frames.extend(glob.glob(os.path.join(cam_folders[0], "*.png")))
    num_frames = len(first_cam_frames)

    # Interaktif önizleme
    print(f"\n📸 {len(cam_folders)} kamera, {num_frames} frame bulundu\n")
    interact(
        preview_mask,
        camera_idx=IntSlider(min=0, max=len(cam_folders)-1, step=1, value=0, description='Kamera:'),
        frame_idx=IntSlider(min=0, max=num_frames-1, step=1, value=0, description='Frame:')
    )

print("\n📝 Maskeler iyi görünüyorsa Cell 6'ya geçin")
print("   Maskeler kötüyse Cell 4'e dönüp parametreleri ayarlayın")

## ⚙️ Cell 6: Eğitim Konfigürasyonu

Eğitim parametrelerini ayarlayın.

In [ ]:
print("="*60)
print("⚙️  Eğitim Konfigürasyonu")
print("="*60)

# Eğitim Presetleri
PRESETS = {
    "quick_test": {
        "iterations": 14000,
        "coarse_iterations": 2000,
        "w_fg": 1.0,
        "w_bg": 0.1,
        "description": "Hızlı test (~30 dk A100, ~1 saat T4)"
    },
    "standard": {
        "iterations": 30000,
        "coarse_iterations": 3000,
        "w_fg": 1.0,
        "w_bg": 0.1,
        "description": "Dengeli kalite/hız (~1.5 saat A100)"
    },
    "high_quality": {
        "iterations": 60000,
        "coarse_iterations": 5000,
        "w_fg": 1.2,
        "w_bg": 0.05,
        "net_width": 128,
        "description": "En iyi kalite (~3-4 saat A100)"
    },
    "fast_motion": {
        "iterations": 45000,
        "coarse_iterations": 3000,
        "w_fg": 1.5,
        "w_bg": 0.1,
        "defor_depth": 2,
        "time_smoothness_weight": 0.005,
        "description": "Dans/aksiyon sahneleri için (~2 saat A100)"
    }
}

# Preset seçin
PRESET = "standard"  # @param ["quick_test", "standard", "high_quality", "fast_motion"]

# Temel parametreler
USE_MASK_LOSS = True  # @param {type:"boolean"}
ITERATIONS = PRESETS[PRESET]["iterations"]  # @param {type:"integer"}
W_FG = PRESETS[PRESET]["w_fg"]  # @param {type:"number"}
W_BG = PRESETS[PRESET]["w_bg"]  # @param {type:"number"}

# Gelişmiş parametreler
BATCH_SIZE = 1
LAMBDA_DSSIM = 0.0
DENSIFY_UNTIL_ITER = 15000
NET_WIDTH = PRESETS[PRESET].get("net_width", 64)
DEFOR_DEPTH = PRESETS[PRESET].get("defor_depth", 1)
TIME_SMOOTHNESS_WEIGHT = PRESETS[PRESET].get("time_smoothness_weight", 0.01)
COARSE_ITERATIONS = PRESETS[PRESET]["coarse_iterations"]

print(f"\n🎯 Seçilen preset: {PRESET}")
print(f"   {PRESETS[PRESET]['description']}")
print("\n📊 Konfigürasyon:")
print(f"   Iterasyon sayısı: {ITERATIONS}")
print(f"   Maske-ağırlıklı loss: {USE_MASK_LOSS}")
if USE_MASK_LOSS:
    print(f"   Ön plan ağırlığı (w_fg): {W_FG}")
    print(f"   Arka plan ağırlığı (w_bg): {W_BG}")

print("\n" + "="*60)
print("✅ Konfigürasyon hazır!")
print("="*60)
print("\n📝 Sonraki adım: Cell 7 ile eğitimi başlatın")

## 🚀 Cell 7: Eğitim

Model eğitimini başlatır. Eğitim lokal diskte yapılır, sonunda Drive'a kopyalanır.

In [ ]:
import time
import shutil

print("="*60)
print("🚀 Eğitim Başlıyor")
print("="*60)

# Komut oluştur
cmd_parts = [
    "python /content/4DGaussians-Enhanced/train.py",
    f"--source_path {LOCAL_DATA}",
    f"--model_path {LOCAL_OUTPUT}",
    f"--iterations {ITERATIONS}",
    f"--coarse_iterations {COARSE_ITERATIONS}",
    f"--batch_size {BATCH_SIZE}",
    f"--lambda_dssim {LAMBDA_DSSIM}",
    f"--densify_until_iter {DENSIFY_UNTIL_ITER}",
    f"--net_width {NET_WIDTH}",
    f"--defor_depth {DEFOR_DEPTH}",
    f"--time_smoothness_weight {TIME_SMOOTHNESS_WEIGHT}",
]

if USE_MASK_LOSS:
    cmd_parts.append("--use_mask_loss")
    cmd_parts.append(f"--w_fg {W_FG}")
    cmd_parts.append(f"--w_bg {W_BG}")

cmd = " ".join(cmd_parts)

print(f"\n📝 Komut:")
print(cmd)
print()

start_time = time.time()

# Eğitimi başlat
!{cmd}

elapsed = time.time() - start_time

print("\n" + "="*60)
print("✅ Eğitim tamamlandı!")
print("="*60)
print(f"⏱️  Süre: {elapsed/3600:.2f} saat")
print(f"📁 Lokal model: {LOCAL_OUTPUT}")

# Drive'a kopyala
print("\n📤 Model Drive'a kopyalanıyor...")
scene_name = os.path.basename(LOCAL_DATA)
drive_output = os.path.join(OUTPUT_BASE, scene_name)

if os.path.exists(drive_output):
    print(f"   Eski çıktı siliniyor: {drive_output}")
    shutil.rmtree(drive_output)

print(f"   Kopyalanıyor: {LOCAL_OUTPUT} -> {drive_output}")
shutil.copytree(LOCAL_OUTPUT, drive_output)
print(f"✅ Model Drive'a kopyalandı: {drive_output}")

print("\n📝 Sonraki adım: Cell 8 ile render yapın")

## 🎥 Cell 8: Render

Eğitilmiş modelden video render eder.

In [ ]:
import glob
from IPython.display import Video, display

print("="*60)
print("🎥 Video Render")
print("="*60)

# Render komutu
render_cmd = f"""python /content/4DGaussians-Enhanced/render.py \
    --source_path {LOCAL_DATA} \
    --model_path {LOCAL_OUTPUT} \
    --iteration {ITERATIONS}"""

print(f"\n📝 Komut:")
print(render_cmd)
print()

!{render_cmd}

# Render edilen videoyu bul
video_files = glob.glob(os.path.join(LOCAL_OUTPUT, "**/*.mp4"), recursive=True)

if video_files:
    print("\n" + "="*60)
    print("✅ Render tamamlandı!")
    print("="*60)
    print(f"\n📹 Video: {video_files[0]}")

    # Videoyu göster
    print("\n📺 Video oynatılıyor...\n")
    display(Video(video_files[0], width=800))

    # Drive'a kopyala
    scene_name = os.path.basename(LOCAL_DATA)
    drive_output = os.path.join(OUTPUT_BASE, scene_name)

    print(f"\n📤 Video Drive'a kopyalanıyor: {drive_output}")
    # Model zaten kopyalandı, sadece render klasörünü güncelle
    render_dir_local = os.path.dirname(video_files[0])
    render_dir_drive = os.path.join(drive_output, os.path.basename(render_dir_local))

    if os.path.exists(render_dir_drive):
        shutil.rmtree(render_dir_drive)
    shutil.copytree(render_dir_local, render_dir_drive)
    print(f"✅ Video Drive'a kopyalandı")
else:
    print("\n⚠️  Video dosyası bulunamadı. Çıktı klasörünü kontrol edin.")

print("\n📝 Sonraki adım: Cell 9 ile PLY export yapın (opsiyonel)")

## 💾 Cell 9: PLY Export (Opsiyonel)

Frame başına 3D Gaussian point cloud'ları export eder.

In [ ]:
EXPORT_PLY = False  # @param {type:"boolean"}

if EXPORT_PLY:
    print("="*60)
    print("💾 PLY Export")
    print("="*60)

    # Export komutu
    export_cmd = f"""python /content/4DGaussians-Enhanced/export_perframe_3DGS.py \
        --source_path {LOCAL_DATA} \
        --model_path {LOCAL_OUTPUT} \
        --iteration {ITERATIONS}"""

    print(f"\n📝 Komut:")
    print(export_cmd)
    print()

    !{export_cmd}

    ply_dir = os.path.join(LOCAL_OUTPUT, "per_frame_ply")
    if os.path.exists(ply_dir):
        ply_files = glob.glob(os.path.join(ply_dir, "*.ply"))

        print("\n" + "="*60)
        print("✅ Export tamamlandı!")
        print("="*60)
        print(f"\n💾 {len(ply_files)} PLY dosyası oluşturuldu")
        print(f"📁 Lokal konum: {ply_dir}")

        # Drive'a kopyala
        scene_name = os.path.basename(LOCAL_DATA)
        drive_output = os.path.join(OUTPUT_BASE, scene_name)
        ply_dir_drive = os.path.join(drive_output, "per_frame_ply")

        print(f"\n📤 PLY dosyaları Drive'a kopyalanıyor: {ply_dir_drive}")
        if os.path.exists(ply_dir_drive):
            shutil.rmtree(ply_dir_drive)
        shutil.copytree(ply_dir, ply_dir_drive)
        print(f"✅ PLY dosyaları Drive'a kopyalandı")
    else:
        print("\n⚠️  Export klasörü bulunamadı. Komut çıktısını kontrol edin.")
else:
    print("⏭️  PLY export atlandı (EXPORT_PLY=False)")
    print("   PLY export yapmak için EXPORT_PLY=True yapın ve tekrar çalıştırın")

print("\n" + "="*60)
print("🎉 Tüm işlemler tamamlandı!")
print("="*60)
print(f"\n📁 Çıktılar: {OUTPUT_BASE}")
print("\n✅ 4D Gaussian modeliniz hazır!")